# Embedding Benchmark — GPU Models (Colab)

Runs the 3 GPU-required embedding models against the GenomIO corpus.
Reuses `experiment_1_embedding_benchmark.py` and `experiment_2_embedding_benchmark.py`
from the local project via `--only`. Persists JSON/CSV/`.npy` outputs to Google Drive.

**GPU model keys (this notebook):** `Caduceus`, `HyenaDNA`, `NT_2500M`.

The previous Evo2_7B / Evo2_1B / AIDO_DNA_7B targets have been replaced in the registry
with CPU-runnable substitutes (`NT_v2_100M`, `NT_v2_250M`, `DNABERT_1`) — these run locally
in `experiment_1/2` without Colab. The replacements fit the thesis narrative better:

- **NT_v2_100M / NT_v2_250M** complete the NT v2 MLM scale curve (50M → 100M → 250M → 500M),
  isolating "size" as the variable while training objective stays constant.
- **DNABERT_1** (6-mer MLM) is the historical predecessor of DNABERT-2 (BPE/MLM) and
  DNABERT-S (BPE/contrastive), letting the analysis attribute the family's improvement to
  contrastive training, not tokenisation.

## Hardware requirements

| Model | Min GPU | Notes |
|---|---|---|
| HyenaDNA | T4+ (any CUDA) | Works on free Colab |
| Caduceus | T4+ | Needs `mamba_ssm` + `causal_conv1d` (Cell 3 installs them with `--no-build-isolation`) |
| NT_2500M (2.5B) | T4+ (16 GB OK in FP16) | |

**Runtime**: Runtime → Change runtime type → GPU (T4 is sufficient).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/genomio_embedding_results'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Results will be saved to: {DRIVE_DIR}')

Mounted at /content/drive
Results will be saved to: /content/drive/MyDrive/genomio_embedding_results


## Cell 2 — Upload project files

In [2]:
from google.colab import files
import zipfile, sys, os

print('Upload genomio_colab_upload.zip from experiments/ in the repo.')
print('It contains: experiment_1_embedding_benchmark.py,')
print('experiment_2_embedding_benchmark.py, and rag_corpus_uniform/')
uploaded = files.upload()

for fname in uploaded:
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/genomio/')

PROJECT_DIR = '/content/genomio'
# If the zip nested everything under a top-level folder, descend into it
entries = os.listdir(PROJECT_DIR)
if len(entries) == 1 and os.path.isdir(os.path.join(PROJECT_DIR, entries[0])):
    inner = os.path.join(PROJECT_DIR, entries[0])
    if os.path.isdir(os.path.join(inner, 'experiments')):
        PROJECT_DIR = inner

sys.path.insert(0, PROJECT_DIR)
EXP_DIR = os.path.join(PROJECT_DIR, 'experiments')
print(f'Project extracted to: {PROJECT_DIR}')
print(f'Experiments dir: {EXP_DIR}')
assert os.path.isfile(os.path.join(EXP_DIR, 'experiment_1_embedding_benchmark.py'))
assert os.path.isfile(os.path.join(EXP_DIR, 'experiment_2_embedding_benchmark.py'))
assert os.path.isdir(os.path.join(PROJECT_DIR, 'rag_corpus_uniform'))

Upload genomio_colab_upload.zip from experiments/ in the repo.
It contains: experiment_1_embedding_benchmark.py,
experiment_2_embedding_benchmark.py, and rag_corpus_uniform/


Saving genomio_colab_upload.zip to genomio_colab_upload.zip
Project extracted to: /content/genomio
Experiments dir: /content/genomio/experiments


## Cell 3 — Install GPU model dependencies

In [ ]:
import subprocess, sys

def run(cmd):
    print(f'\n$ {cmd}')
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(p.stdout[-1500:])
    if p.returncode != 0:
        print('STDERR:', p.stderr[-1500:])
    return p.returncode == 0

# Standard installs (most are quick wheels)
for step in [
    'pip install -q transformers torch scikit-learn numpy tqdm umap-learn biopython pandas seaborn matplotlib multimolecule',
    'pip install -q packaging ninja wheel setuptools',
    'pip install -q sentencepiece protobuf tiktoken',
    'pip install -q --upgrade tokenizers',
    'pip install -q --no-build-isolation causal-conv1d',
    'pip install -q --no-build-isolation mamba-ssm',
]:
    ok = run(step)
    print('OK' if ok else 'FAILED')

# flash-attn — source build needs ~60 min and exhausts Colab memory.
# Use a prebuilt wheel matching the active torch + Python + CUDA + ABI.
print('\n=== flash-attn prebuilt wheel ===')
import torch
torch_ver = torch.__version__.split('+')[0]
torch_short = '.'.join(torch_ver.split('.')[:2])  # "2.5" / "2.6"
py = f'cp{sys.version_info.major}{sys.version_info.minor}'  # "cp310" / "cp311"
cuda = (torch.version.cuda or '12.4').replace('.', '')[:3]  # "121" / "124"
# Try a few likely-matching wheels in order
wheel_candidates = [
    f'https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch{torch_short}cxx11abiFALSE-{py}-{py}-linux_x86_64.whl',
    f'https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.3/flash_attn-2.7.3+cu12torch{torch_short}cxx11abiFALSE-{py}-{py}-linux_x86_64.whl',
    f'https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch{torch_short}cxx11abiFALSE-{py}-{py}-linux_x86_64.whl',
]
flash_ok = False
for url in wheel_candidates:
    print(f'\nTrying: {url}')
    if run(f'pip install -q {url}'):
        flash_ok = True
        break
if not flash_ok:
    print('No prebuilt wheel matched — Evo2 will be skipped.')

# Remaining installs
for step in [
    'cd /content && git clone --depth=1 https://github.com/kuleshov-group/caduceus.git || true && pip install -q --no-build-isolation ./caduceus || echo "Caduceus local pkg not needed; HF code works via trust_remote_code"',
    'pip install -q --upgrade git+https://github.com/huggingface/transformers.git',
    'cd /content && git clone --recurse-submodules --depth=1 https://github.com/ArcInstitute/evo2.git || true && pip install -q ./evo2',
]:
    ok = run(step)
    print('OK' if ok else 'FAILED')

print('\n=== Final import check ===')
for mod in ('mamba_ssm', 'causal_conv1d', 'flash_attn', 'evo2', 'multimolecule', 'sentencepiece'):
    try:
        __import__(mod)
        print(f'  {mod}: OK')
    except Exception as e:
        print(f'  {mod}: FAIL ({type(e).__name__}: {str(e)[:120]})')

## Cell 4 — Verify GPU

In [4]:
import torch
assert torch.cuda.is_available(), 'No GPU detected. Change Runtime > Runtime type > GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## Cell 5 — Run Experiment 1 for GPU models

In [ ]:
import subprocess, json, shutil, gc
import pandas as pd
import torch

GPU_KEYS = ['Caduceus', 'HyenaDNA', 'NT_2500M']
EXP1_SCRIPT = os.path.join(EXP_DIR, 'experiment_1_embedding_benchmark.py')
EXP1_CSV    = os.path.join(EXP_DIR, 'experiment_1_results.csv')

def print_diagnostic_lines(combined_output):
    """Print only the lines that explain success/failure, hiding the noisy
    HuggingFace download progress and HTTP request logs."""
    for line in combined_output.splitlines():
        if any(s in line for s in (
            'failed:', 'Traceback', 'Error', 'ImportError', 'ModuleNotFoundError',
            'AttributeError', 'RuntimeError', 'CUDA', 'OOM', 'killed',
            'Loading', 'silhouette_score', 'embedding_dim',
            'SKIPPED', 'cannot', 'Could', 'not a valid', 'is not',
            'CADUCEUS PATCH', 'Neutralised', 'hasattr check'
        )):
            print('  ' + line[:250])

for key in GPU_KEYS:
    print(f'\n{"="*60}\nExperiment 1 — {key}\n{"="*60}')
    p = subprocess.run(
        ['python3', EXP1_SCRIPT, '--only', key, '--force'],
        cwd=PROJECT_DIR, check=False, capture_output=True, text=True
    )
    torch.cuda.empty_cache(); gc.collect()
    print_diagnostic_lines(p.stdout + '\n' + p.stderr)
    if p.returncode != 0:
        print(f'  {key}: subprocess returned {p.returncode}')
        continue
    if not os.path.isfile(EXP1_CSV):
        print(f'  {key}: CSV not produced — skipping save')
        continue
    df = pd.read_csv(EXP1_CSV)
    matches = df[df['model'] == key]
    if len(matches) == 0:
        print(f'  {key}: no row in CSV (model likely skipped or failed)')
        continue
    row = matches.iloc[-1].to_dict()
    out_path = os.path.join(DRIVE_DIR, f'exp1_{key}.json')
    with open(out_path, 'w') as f:
        json.dump(row, f, indent=2)
    print(f'  {key}: saved row to {out_path}')

print('\nExperiment 1 GPU runs complete.')

## Cell 6 — Run Experiment 2 for GPU models (+ save embeddings)

In [ ]:
EXP2_SCRIPT = os.path.join(EXP_DIR, 'experiment_2_embedding_benchmark.py')
EXP2_CSV    = os.path.join(EXP_DIR, 'experiment_2_results.csv')
LABELS_NPY  = os.path.join(EXP_DIR, 'experiment_2_labels.npy')
DRIVE_LABELS = os.path.join(DRIVE_DIR, 'labels_1000.npy')

for key in GPU_KEYS:
    print(f'\n{"="*60}\nExperiment 2 — {key}\n{"="*60}')
    p = subprocess.run(
        ['python3', EXP2_SCRIPT, '--only', key, '--force'],
        cwd=PROJECT_DIR, check=False, capture_output=True, text=True
    )
    torch.cuda.empty_cache(); gc.collect()
    print_diagnostic_lines(p.stdout + '\n' + p.stderr)
    if p.returncode != 0:
        print(f'  {key}: subprocess returned {p.returncode}')
        continue
    if not os.path.isfile(EXP2_CSV):
        print(f'  {key}: CSV not produced — skipping save')
        continue
    df = pd.read_csv(EXP2_CSV)
    matches = df[df['model'] == key]
    if len(matches) == 0:
        print(f'  {key}: no row in CSV (model likely skipped or failed)')
        continue
    row = matches.iloc[-1].to_dict()
    json_path = os.path.join(DRIVE_DIR, f'exp2_{key}.json')
    with open(json_path, 'w') as f:
        json.dump(row, f, indent=2)
    print(f'  {key}: saved row to {json_path}')
    # Copy embeddings to Drive
    src_emb = os.path.join(EXP_DIR, f'experiment_2_embeddings_{key}.npy')
    if os.path.isfile(src_emb):
        dst_emb = os.path.join(DRIVE_DIR, f'embeddings_1000_{key}.npy')
        shutil.copyfile(src_emb, dst_emb)
        print(f'  {key}: embeddings copied to {dst_emb}')
    # Save labels (write-once)
    if os.path.isfile(LABELS_NPY) and not os.path.isfile(DRIVE_LABELS):
        shutil.copyfile(LABELS_NPY, DRIVE_LABELS)
        print(f'  labels copied to {DRIVE_LABELS}')

print('\nExperiment 2 GPU runs complete.')

## Cell 7 — Consolidate GPU results into CSVs

In [7]:
import glob

COLUMNS = ['model', 'embedding_dim', 'intra_species_sim', 'inter_species_dist',
           'silhouette_score', 'inference_time_per_seq', 'total_inference_time_s']

for tag in ('exp1', 'exp2'):
    json_files = sorted(glob.glob(os.path.join(DRIVE_DIR, f'{tag}_*.json')))
    rows = []
    for jf in json_files:
        with open(jf) as f:
            rows.append(json.load(f))
    if not rows:
        print(f'{tag}: no JSON results found')
        continue
    df = pd.DataFrame(rows)
    df = df[[c for c in COLUMNS if c in df.columns]]
    out_csv = os.path.join(DRIVE_DIR, f'{tag}_gpu.csv')
    df.to_csv(out_csv, index=False)
    print(f'{tag}: wrote {out_csv} with {len(df)} rows')
    print(df.to_string(index=False))

exp1: wrote /content/drive/MyDrive/genomio_embedding_results/exp1_gpu.csv with 2 rows
   model  embedding_dim  intra_species_sim  inter_species_dist  silhouette_score  inference_time_per_seq  total_inference_time_s
HyenaDNA            256             0.9934              0.0541            0.2372                   2.683                  13.416
NT_2500M           2560             0.9720              0.1280            0.1723                  11.637                  58.184
exp2: wrote /content/drive/MyDrive/genomio_embedding_results/exp2_gpu.csv with 2 rows
   model  embedding_dim  intra_species_sim  inter_species_dist  silhouette_score  inference_time_per_seq  total_inference_time_s
HyenaDNA            256             0.9877               0.042           -0.0386                   0.029                  28.823
NT_2500M           2560             0.9375               0.105            0.0090                   0.035                  35.198


## Cell 8 — Final checklist

In [8]:
import os

expected = [
    'exp1_gpu.csv', 'exp2_gpu.csv',
    'labels_1000.npy',
    *[f'exp1_{k}.json' for k in GPU_KEYS],
    *[f'exp2_{k}.json' for k in GPU_KEYS],
    *[f'embeddings_1000_{k}.npy' for k in GPU_KEYS],
]

print(f'Checking {DRIVE_DIR}\n')
ok_count = 0
for name in expected:
    path = os.path.join(DRIVE_DIR, name)
    if os.path.isfile(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'  [OK]   {name}  ({size_kb:.1f} KB)')
        ok_count += 1
    else:
        print(f'  [MISS] {name}')

print(f'\n{ok_count}/{len(expected)} expected files present.')
print('\nNext step (local): download contents of this Drive folder into experiments/')
print('Then run: python3 experiments/plot_species_separation.py')

Checking /content/drive/MyDrive/genomio_embedding_results

  [OK]   exp1_gpu.csv  (0.2 KB)
  [OK]   exp2_gpu.csv  (0.2 KB)
  [OK]   labels_1000.npy  (4.0 KB)
  [MISS] exp1_Caduceus.json
  [OK]   exp1_HyenaDNA.json  (0.2 KB)
  [OK]   exp1_NT_2500M.json  (0.2 KB)
  [MISS] exp1_Evo2_7B.json
  [MISS] exp1_Evo2_1B.json
  [MISS] exp1_AIDO_DNA_7B.json
  [MISS] exp2_Caduceus.json
  [OK]   exp2_HyenaDNA.json  (0.2 KB)
  [OK]   exp2_NT_2500M.json  (0.2 KB)
  [MISS] exp2_Evo2_7B.json
  [MISS] exp2_Evo2_1B.json
  [MISS] exp2_AIDO_DNA_7B.json
  [MISS] embeddings_1000_Caduceus.npy
  [OK]   embeddings_1000_HyenaDNA.npy  (1000.1 KB)
  [OK]   embeddings_1000_NT_2500M.npy  (10000.1 KB)
  [MISS] embeddings_1000_Evo2_7B.npy
  [MISS] embeddings_1000_Evo2_1B.npy
  [MISS] embeddings_1000_AIDO_DNA_7B.npy

9/21 expected files present.

Next step (local): download contents of this Drive folder into experiments/
Then run: python3 experiments/plot_species_separation.py
